# Purpose:
- Modified dev_initial.ipynb to process all Thyme natural images sessions.
    - Using events, not filtered_events (because bod does not support filtered_events, and it is actually cheating)
- Without pupil for now.
    - Integrate them in another notebook when it's ready.
    - Also consider "strategy"

In [2]:
from DesignMatrix import DesignMatrix
from comb.behavior_ophys_dataset import BehaviorOphysDataset, BehaviorMultiplaneOphysDataset
from comb.behavior_session_dataset import BehaviorSessionDataset
import os
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import json

# notebook dev
%load_ext autoreload
%autoreload 2
%matplotlib inline
import capsule_utils
import load_data
import design_matrix_tools as dmtools
import kernel_tools as ktools

In [17]:
def build_input_kernels():
    kernels = {
        'intercept':    {'feature':'intercept',   'type':'continuous',    'length':0,     'offset':0,     'num_weights':None, 'dropout':True, 'text': 'constant value'},
        'hits':         {'feature':'hit',         'type':'discrete',      'length':1.5,   'offset':0,    'num_weights':None, 'dropout':True, 'text': 'lick to image change'},
        'misses':       {'feature':'miss',        'type':'discrete',      'length':1.5,   'offset':0,    'num_weights':None, 'dropout':True, 'text': 'no lick to image change'},
        'passive_change':   {'feature':'passive_change','type':'discrete','length':1.5,   'offset':0,    'num_weights':None, 'dropout':True, 'text': 'passive session image change'},
        #'hits':         {'feature':'hit',         'type':'discrete',      'length':.75,   'offset':0,    'num_weights':None, 'dropout':True, 'text': 'lick to image change'},
        #'misses':       {'feature':'miss',        'type':'discrete',      'length':.75,   'offset':0,    'num_weights':None, 'dropout':True, 'text': 'no lick to image change'},
        #'passive_change':   {'feature':'passive_change','type':'discrete','length':.75,   'offset':0,    'num_weights':None, 'dropout':True, 'text': 'passive session image change'},
        #'post-hits':    {'feature':'hit',         'type':'discrete',      'length':1.5,   'offset':0.75,    'num_weights':None, 'dropout':True, 'text': 'lick to image change'},
        #'post-misses':  {'feature':'miss',        'type':'discrete',      'length':1.5,   'offset':0.75,    'num_weights':None, 'dropout':True, 'text': 'no lick to image change'},
        #'post-passive_change': {'feature':'passive_change','type':'discrete','length':1.5,   'offset':0.75,    'num_weights':None, 'dropout':True, 'text': 'passive session image change'},
        'omissions':        {'feature':'omissions',   'type':'discrete',  'length':1.5,      'offset':0,     'num_weights':None, 'dropout':True, 'text': 'image was omitted'},
        #'omissions':        {'feature':'omissions',   'type':'discrete',  'length':0.75,      'offset':0,     'num_weights':None, 'dropout':True, 'text': 'image was omitted'},
        #'post-omissions':   {'feature':'omissions',   'type':'discrete',  'length':2.25,   'offset':0.75,  'num_weights':None, 'dropout':True, 'text': 'images after omission'},
        'each-image':   {'feature':'each-image',  'type':'discrete',      'length':0.75,  'offset':0,     'num_weights':None, 'dropout':True, 'text': 'image presentation'},
        'running':      {'feature':'running',     'type':'continuous',    'length':2,     'offset':-1,    'num_weights':None, 'dropout':True, 'text': 'normalized running speed'},
        # 'pupil':        {'feature':'pupil',       'type':'continuous',    'length':2,     'offset':-1,    'num_weights':None, 'dropout':True, 'text': 'Z-scored pupil diameter'},
        'licks':        {'feature':'licks',       'type':'discrete',      'length':2,     'offset':-1,    'num_weights':None, 'dropout':True, 'text': 'mouse lick'},
        #'false_alarms':     {'feature':'false_alarm',   'type':'discrete','length':5.5,   'offset':-1,    'num_weights':None, 'dropout':True, 'text': 'lick on catch trials'},
        #'correct_rejects':  {'feature':'correct_reject','type':'discrete','length':5.5,   'offset':-1,    'num_weights':None, 'dropout':True, 'text': 'no lick on catch trials'},
        #'time':         {'feature':'time',        'type':'continuous',    'length':0,     'offset':0,    'num_weights':None,  'dropout':True, 'text': 'linear ramp from 0 to 1'},
        #'beh_model':    {'feature':'beh_model',   'type':'continuous',    'length':.5,    'offset':-.25, 'num_weights':None,  'dropout':True, 'text': 'behavioral model weights'},
        #'lick_bouts':   {'feature':'lick_bouts',  'type':'discrete',      'length':4,     'offset':-2,   'num_weights':None,  'dropout':True, 'text': 'lick bout'},
        #'lick_model':   {'feature':'lick_model',  'type':'continuous',    'length':2,     'offset':-1,   'num_weights':None,  'dropout':True, 'text': 'lick probability from video'},
        #'groom_model':  {'feature':'groom_model', 'type':'continuous',    'length':2,     'offset':-1,   'num_weights':None,  'dropout':True, 'text': 'groom probability from video'},
    }
    ## add face motion energy PCs
    # for PC in range(5):
    #     kernels['face_motion_PC_{}'.format(PC)] = {'feature':'face_motion_PC_{}'.format(PC), 'type':'continuous', 'length':2, 'offset':-1, 'dropout':True, 'text':'PCA from face motion videos'}
    
    return kernels


def get_bod_list(raw_path):
    session_name = str(raw_path).split('/')[-1]
    data_dir = Path(raw_path).parent
    processed_path = list(data_dir.glob(f'{session_name}_processed*'))[0]
    
    opids = []
    for plane_folder in processed_path.glob("*"):
        if plane_folder.is_dir() and not plane_folder.stem.startswith("nextflow"):
            opid = plane_folder.stem
            opids.append(opid)

    bod_list = []
    for opid in opids:
        bod = load_data.load_plane_data(session_name, opid=opid)
        bod = capsule_utils.add_trials_to_bod(bod)
        bod_list.append(bod)
    return bod_list, session_name


def generate_resonpse_and_design_matrices(bod_list):
    ###################################################
    # Generate run_params, design matrix, and responses
    run_params = {'data_type': 'events'}
    input_kernel_dict = build_input_kernels()

    response_list = []
    run_params_list = []
    for bod in bod_list:
        run_params = ktools.process_kernels(input_kernel_dict, run_params, bod)
        response, run_params = load_data.extract_and_annotate_ophys_plane(bod, run_params)
        response_list.append(response)
        run_params_list.append(run_params)
    
    # assert all run_params are the same
    assert all([run_params == run_params_list[0] for run_params in run_params_list])
    timestamps = [r['timestamps'] for r in response_list]
    assert all([(ts == timestamps[0]).all() for ts in timestamps])
    timebins = [r['time_bins'] for r in response_list]
    assert all([(tb == timebins[0]).all() for tb in timebins])
    ophys_frame_rates = [r['ophys_frame_rate'] for r in response_list]
    assert ~np.diff(ophys_frame_rates).any()
    
    # run params should be the same across planes
    run_params = run_params_list[0]
    run_params['input_kernel_dict'] = input_kernel_dict  # add for future use

    # response session array from concatenating all planes
    response_session_arr = xr.concat([r['response_arr'] for r in response_list], dim='cell_roi_id')

    # set up a response dictionary for adding kernels
    response = {}
    response['response_arr'] = response_session_arr
    response['timestamps'] = timestamps[0]
    response['time_bins'] = timebins[0]
    response['ophys_frame_rate'] = ophys_frame_rates[0]

    # response archive for saving
    response_archive = [r['stimulus_interpolation'] for r in response_list]

    # response matrix for saving
    response_matrix = response['response_arr']

    # response info dictionary for saving
    response_info = {}
    response_info['timestamps'] = response['timestamps']
    response_info['time_bins'] = response['time_bins']
    response_info['ophys_frame_rate'] = response['ophys_frame_rate']
    
    # add kernels to the designMatrix class
    design = DesignMatrix(response['timestamps'], response['ophys_frame_rate'])
    dmtools.add_kernels(design, run_params, bod, response) # need the trace array for kernels like population mean or PC1. 
    
    # assertion
    X = design.get_X()
    assert X.shape[0] == response['response_arr'].shape[0]

    return run_params, design, response_matrix, response_info, response_archive


def run_and_save_session(raw_path, save_path=Path('/root/capsule/scratch/')):
    bod_list, session_name = get_bod_list(raw_path)
    run_params, design, response_matrix, response_info, response_archive = \
        generate_resonpse_and_design_matrices(bod_list)
    
    ###############
    # Save them all    
    session_save_path_desmat = save_path / f'design_matrix_{session_name}_00' # 00 just for the test
    session_save_path_desmat.mkdir(parents=True, exist_ok=True)

    data_type = run_params['data_type']
    session_save_path_resmat = save_path / f'response_matrix_{session_name}_{data_type}'
    session_save_path_resmat.mkdir(parents=True, exist_ok=True)

    # serialize sets in run_params
    run_params_keys = list(run_params.keys())
    set_inds = np.where([type(run_params[key])==set for key in run_params_keys])[0]
    for i in set_inds:
        key = run_params_keys[i]
        run_params[key] = list(run_params[key])

    run_params_fn = session_save_path_desmat / 'run_params.json'
    with open(run_params_fn, 'w') as f:
        json.dump(run_params, f, indent=4)

    design_matrix_fn = session_save_path_desmat / 'design_matrix.nc'
    X = design.get_X()
    X.to_netcdf(design_matrix_fn)

    dm_unstd_features_fn = session_save_path_desmat / 'unstd_features.npy'
    np.save(dm_unstd_features_fn, design.unstd_features)



    response_matrix_fn = session_save_path_resmat / 'response_matrix.nc'
    response_matrix.to_netcdf(response_matrix_fn)

    response_info_fn = session_save_path_resmat / 'response_info.npy'
    np.save(response_info_fn, response_info)
        
    response_archive_fn = session_save_path_resmat / 'response_archive.npy'
    np.save(response_archive_fn, response_archive)

In [15]:
data_dir = '/root/capsule/data'
data_folders = [d for d in glob.glob(data_dir + '/*') if Path(d).is_dir()]
raw_paths = [d for d in data_folders if ('processed' not in d.split('/')[-1]) and
                 ('dlc-eye' not in d.split('/')[-1]) and
                 ('multiplane-ophys' in d.split('/')[-1]) and 
                 ('stimuli' not in d.split('/')[-1])]


AttributeError: 'str' object has no attribute 'stem'

In [21]:
for i, raw_path in enumerate(raw_paths):
    session_name = raw_path.split('/')[-1]
    print(f'Processing {session_name} ({i}/{len(raw_paths)})')
    run_and_save_session(raw_path)

Processing multiplane-ophys_736963_2024-08-12_09-17-19 (0/11)


/comb/src/comb/processing/stimulus/stimulus_processing.py:802: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  is_change = is_change.fillna(False)
/comb/src/comb/processing/stimulus/presentations.py:514: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  omitted = df["omitted"].fillna(False)
/root/capsule/code/capsule_utils.py:37: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  change_times = stimulus_presentations.query('is_change').start_time.values
/comb/src

Using events traces
Interpolating neural signal onto stimulus aligned timestamps
Using events traces
Interpolating neural signal onto stimulus aligned timestamps
Using events traces
Interpolating neural signal onto stimulus aligned timestamps
Using events traces
Interpolating neural signal onto stimulus aligned timestamps
Using events traces
Interpolating neural signal onto stimulus aligned timestamps
Using events traces
Interpolating neural signal onto stimulus aligned timestamps
Using events traces
Interpolating neural signal onto stimulus aligned timestamps
Using events traces
Interpolating neural signal onto stimulus aligned timestamps
    Adding kernel: intercept
    Adding kernel: hits
    Adding kernel: misses
    Adding kernel: passive_change
	Error encountered while adding kernel for passive_change. Attemping to continue without this kernel.
	Passive Change kernel cant be added to active sessions
    Adding kernel: omissions
    Adding kernel: running
                 : Mean C

/root/capsule/code/design_matrix_tools.py:299: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  feature_times = bod.stimulus_presentations.query('omitted')['start_time'].values


    Adding kernel: im075
    Adding kernel: im106
The following kernels failed to be added to the model: 
{'passive_change'}

The following dropouts failed to be added to the model: 
{'single-passive_change'}



...

Processing multiplane-ophys_736963_2024-08-13_08-57-29 (10/11)


/comb/src/comb/processing/stimulus/stimulus_processing.py:802: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  is_change = is_change.fillna(False)
/comb/src/comb/processing/stimulus/presentations.py:514: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  omitted = df["omitted"].fillna(False)
/root/capsule/code/capsule_utils.py:37: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  change_times = stimulus_presentations.query('is_change').start_time.values
/comb/src

Using events traces
Interpolating neural signal onto stimulus aligned timestamps
Using events traces
Interpolating neural signal onto stimulus aligned timestamps
Using events traces
Interpolating neural signal onto stimulus aligned timestamps
Using events traces
Interpolating neural signal onto stimulus aligned timestamps
Using events traces
Interpolating neural signal onto stimulus aligned timestamps
Using events traces
Interpolating neural signal onto stimulus aligned timestamps
Using events traces
Interpolating neural signal onto stimulus aligned timestamps
Using events traces
Interpolating neural signal onto stimulus aligned timestamps
    Adding kernel: intercept
    Adding kernel: hits
    Adding kernel: misses
    Adding kernel: passive_change
	Error encountered while adding kernel for passive_change. Attemping to continue without this kernel.
	Passive Change kernel cant be added to active sessions
    Adding kernel: omissions
    Adding kernel: running
                 : Mean C

/root/capsule/code/design_matrix_tools.py:299: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  feature_times = bod.stimulus_presentations.query('omitted')['start_time'].values


    Adding kernel: im106
The following kernels failed to be added to the model: 
{'passive_change'}

The following dropouts failed to be added to the model: 
{'single-passive_change'}

